# Arithmetic Data Types

Quantum arithmetic data types are quantum analogues of classical integer and real numbers. 

The Qubits data type we've used in the earlier katas represents "raw" qubit registers - arrays of qubits that can be interpreted as bit strings, numbers in any encoding, or any other data type. However, fault-tolerant quantum algorithms often involve qubit registers that encode a numeric value and performing arithmetic computations on superpositions of numeric values coherently. Same as introducing `int` and `float` data types simplifies expressing numeric calculations in classical programming, having specialized quantum data types simplifies such calculations in quantum programming.

In this kata, we will explore Workbench representation of integer and real numbers, both signed and unsigned, and implement simple arithmetic operations acting on different data types.

> Note that the naive addition and subtraction algorithms we'll implement here are the most straightforward implementations of these arithmetic operations, but not the most efficient ones! We will learn several adder implementations and compare their resource requirements in another kata.

**This kata covers the following topics:**

- Big-endian and little-endian binary encodings for representing numbers
- Two's complement binary encoding for representing negative numbers
- Workbench representation of unsigned and signed integers
- Naive addition and subtraction algorithms that correspond to the `NaiveAdd` Qubrick in Workbench
- Testing reversible computations involving arithmetic data types

**What you should know to start working on this kata:**

- Fundamental quantum concepts
- Controlled gates
- Oracles, in particular marking oracles

## Part 1. Unsigned integers (QUInt)

In the first part of this kata, we'll discuss the representation of unsigned integers in Workbench data type `QUInt` and learn to add and subtract unsigned integers.

### Little-endian binary encoding for unsigned integers

There are two main ways to represent an integer as a bit string: big-endian and little-endian. Let's consider a bit string $x_0x_1...x_{N-1}$. How can we convert it to an unsigned integer?

In big-endian encoding, the leftmost bit $x_0$ is the _most significant_, and the rightmost bit $x_{N-1}$ is the _least significant_. The number represented by this bit string is then interpreted as 

$$x = 2^{N-1} x_0 + 2^{N-2} x_1 + ... +  2^2 x_{N-3} + 2 x_{N-2} + x_{N-1}$$

Big-endian encoding is used in many common scenarios, including writing numbers (decimal or binary) in arithmetic or in Python. For example, Python binary literal `0b10110` assumes big-endian notation: $10110_{BE} = 2^4 + 2^2 + 2^1 = 22$.

In little-endian encoding, however, the leftmost bit $x_0$ is the _least significant_, and the rightmost bit $x_{N-1}$ is the _most significant_. The number represented by this bit string is then interpreted as 

$$x = x_0 + 2 x_1 + 2^2 x_2 + ... + 2^{N-1} x_{N-1}$$

The same example bit string `10110` in little-endian would encode $10110_{LE} = 2^0 + 2^2 + 2^3 = 13$.

**Workbench uses little-endian encoding: the least significant bit is stored first.**

This encoding affects any APIs that rely on a mapping of integers to bit strings and vice versa, including writing a basis state to a quantum register and reading it out.

Let's practice manipulating integers represented in little-endian encoding by implementing a naive adder for such integers.

> In each problem, you are given a description of a classical function that should be implemented as a reversible computation. As usual in this case, the effect of the unitary operation on a superposition state is defined via its effect on basis states. Given a basis state $\ket{x}$ (and, later, basis states of several inputs), the unitary should change that basis state to a different one following the classical function, without introducing a relative phase in the process.

### Problem 1. Increment by 1

**Input:** 
A QUInt register of length $N$ in an arbitrary superposition state $\ket{a}$.

**Goal:** 
Increment the number $a$ stored in the register modulo $2^N$, that is, transform the state to $\ket{(a + 1) \textrm{ mod } 2^N}$.

In [ ]:
from psiqdk.workbench import QUInt
from test_ArithmeticDataTypes import problem

@problem
def increment(a: QUInt) -> None:
    # Write your code here
    ...

### Problem 2. Increment by a power of 2

**Inputs:** 

1. A QUInt register of length $N$ in an arbitrary superposition state $\ket{a}$.
2. An integer $p$ between $0$ and $N - 1$, inclusive.

**Goal:** 
Increment the number $a$ stored in the register by $2^p$ modulo $2^N$, that is, transform the state to $\ket{(a + 2^p) \textrm{ mod } 2^N}$.

In [ ]:
from psiqdk.workbench import QUInt
from test_ArithmeticDataTypes import problem

@problem
def increment_power(a: QUInt, p: int) -> None:
    # Write your code here
    ...

### Problem 3. Increment by a constant

**Inputs:** 

1. A QUInt register of length $N$ in an arbitrary superposition state $\ket{a}$.
2. An integer $b$ between $0$ and $2^N - 1$, inclusive.

**Goal:** 
Increment the number $a$ stored in the register by $b$ modulo $2^N$, that is, transform the state to $\ket{(a + b) \textrm{ mod } 2^N}$.

In [ ]:
from psiqdk.workbench import QUInt
from test_ArithmeticDataTypes import problem

@problem
def increment_constant(a: QUInt, b: int) -> None:
    # Write your code here
    ...

### Problem 4. Add two unsigned integers

**Inputs:** 

1. A QUInt register of length $N$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register of length $M \le N$ in an arbitrary superposition state $\ket{b}$.

**Goal:** 
Add the number $b$ to the number $a$ modulo $2^N$. Leave the register $\ket{b}$ unchanged. In other words, implement the following transformation: 

$$\ket{a}\ket{b} \rightarrow \ket{(a + b) \textrm{ mod } 2^N}\ket{b}$$

> Note that for this task we'll switch to implementing the solutions as Qubricks instead of functions. You'll see why in the next task! 
>
> Implementing a Qubrick solution is very similar to implementing a simple function: you're given a skeleton class with one or several methods to implement.
>
> * The main method of a Qubrick class is `_compute`: this is the method you'll call to run the computation performed by this Qubrick in place of a function call.
> * In this case, the Qubrick has an additional private method `_increment` which implements controlled variant of the `increment()` function from problem 1. It is not supposed to be called from outside the Qubrick, but it can be used by the `_compute` implementation.
>
> You can read more about implementing Qubricks in [Qubricks tutorial](https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Qubricks.html).

In [ ]:
from psiqdk.workbench import Qubits, QUInt, Qubrick
from test_ArithmeticDataTypes import problem

@problem
class NaiveAdd(Qubrick):
    def _increment(self, a: QUInt, cond: Qubits) -> None:
        """Increment a QUInt register with a quantum control."""
        # Write your code here
        ...

    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

### Problem 5. Subtract two unsigned integers

**Inputs:** 

1. A QUInt register of length $N$ in an arbitrary superposition state $\ket{a}$.
2. A QUInt register of length $M \le N$ in an arbitrary superposition state $\ket{b}$.

**Goal:** 
Subtract the number $b$ from the number $a$ modulo $2^N$. Leave the register $\ket{b}$ unchanged. In other words, implement the following transformation: 

$$\ket{a}\ket{b} \rightarrow \ket{(a - b) \textrm{ mod } 2^N}\ket{b}$$

<details>
<summary><strong>Need a hint?</strong></summary>
You can think of subtraction as the inverse of addition. Is there a quantum computing concept that reflects this? How to express it in Workbench? You might find <a href="https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Qubricks-Uncomputation.html">this tutorial</a> useful.
</details>

In [ ]:
from psiqdk.workbench import QUInt
from test_ArithmeticDataTypes import problem

@problem
class NaiveSubtract(NaiveAdd):
    # Write your code here
    ...

### Demo: Testing reversible computations

Now that we have implemented addition and subtraction for unsigned integers, let's pause and consider how we can test our implementations. The katas test your solutions for you, of course, but once you're writing your own code, testing it becomes critically important.

The technique we'll use for testing addition and subtraction can be reused for pretty much any reversible computation - quantum computation that evaluates a classical function. It follows the approach introduced in the Oracles kata:

1. Write the quantum code for the reversible computation you want to test.
2. Write the classical function this computation implements.
3. Run the quantum code on each input basis state individually (or, if the space is too large, on each of a subset of basis states). You can use Workbench [bit vector simulator](https://construct.psiquantum.com/docs/psiqdk-workbench/new-tutorials/Simulating-WB-Programs.html#bit-vector-simulator) (filter `>>bit-sim>>`, configured in filter preset `BIT_DEFAULT`), designed specifically for simulating reversible computations, to make these simulations efficient.
4. Check the results of quantum code against those of the classical code:
     - The quantum registers that store the result should store the correct basis state.
     - The quantum registers that are used only as input should remain unchanged.
   
   In our example, addition and subtraction act in-place, so register $a$ should store the function value and register $b$ should be unchanged.
5. If the quantum code acts correctly on all basis states, and you didn't use any measurements or gates that can introduce superposition or relative phases, it will also act correctly on superposition inputs!

The following code shows a simple testing harness that you can use to test `NaiveAdd` and `NaiveSubtract` from the previous problems.

In [ ]:
from typing import Callable
from psiqdk.workbench import QPU, Qubits, QUInt, Qubrick
from psiqdk.workbench.filter_presets import BIT_DEFAULT

class NaiveAdd(Qubrick):
    def _increment(self, a: QUInt, cond: Qubits) -> None:
        """Increment a QUInt register with a quantum control."""
        for ind in range(len(a) - 1, 0, -1):
            a[ind].x(cond=a[:ind] | cond)
        a[0].x(cond=cond)
    
    def _compute(self, a: QUInt, b: QUInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        m = len(b)
        for ind in range(m):
            self._increment(a[ind:], b[ind])

class NaiveSubtract(NaiveAdd):
    def __init__(self, **kwargs):
        super().__init__(dagger=True, **kwargs)

def f_add(a: int, b: int, n: int) -> int:
    """Add two unsigned integers modulo 2^n."""
    return (a + b) % (2 ** n)

def f_subtract(a: int, b: int, n: int) -> int:
    """Subtract two unsigned integers modulo 2^n."""
    return (2 ** n + a - b) % (2 ** n)


def run_test_reversible(n_bits: int, qubrick: type[Qubrick], fun: Callable):
    """Run a reversible computation that implements a function of two inputs
    on all possible inputs and check the results against the classical results."""
    # Create QPU instance to run code on the bit simulator
    qpu = QPU(filters=BIT_DEFAULT)

    # Create a Qubrick instance
    qbk = qubrick()

    # Iterate over all possible pairs of inputs
    for input_a in range(2 ** n_bits):
        for input_b in range(2 ** n_bits):
            # Reset QPU and allocate the registers
            qpu.reset(2 * n_bits)
            a = QUInt(n_bits, "a", qpu)
            b = QUInt(n_bits, "b", qpu)

            # Initialize the registers with inputs
            a.write(input_a)
            b.write(input_b)

            # Call the reversible computation
            qbk.compute(a, b)

            # Get the results of the quantum computation
            res_a = a.read()
            res_b = b.read()

            # Evaluate classical function on the classical inputs
            res_classical = fun(input_a, input_b, n_bits)

            # Compare the results of classical and quantum computations
            prefix = f"Error for a={input_a}, b={input_b}: "
            if res_a != res_classical:
                raise Exception(f"{prefix}expected result a={res_classical}, got {res_a}")
            if res_b != input_b:
                raise Exception(f"{prefix}the state of the register b changed to {res_b}")


# Run the test for two Qubricks on a variety of input sizes
for (qubrick, fun) in [(NaiveAdd, f_add), 
                       (NaiveSubtract, f_subtract)]:
    for n_bits in range(2, 5):
        run_test_reversible(n_bits, qubrick, fun)
print("Tests passed!")

## Part 2. Signed integers (QInt)

In the second part of this kata, we'll discuss the representation of signed integers in Workbench data type `QInt` and learn to add and subtract signed integers.

### Two's complement encoding for signed integers

How can we extend the little-endian encoding for unsigned to handle signed integers consistently? 

The definition of the little-endian encoding is

$$x_0x_1...x_{N-1} = \sum_{s=0}^{N-1} x_s 2^s$$

This formula does not seem to let us access negative numbers at all. However, we can use the fact that adders become modular when the bit size of the number is fixed to $N$: 

$$\underbrace{11...1}_N + 1 = \underbrace{00...0}_N$$

This suggests that we should try and interpret the binary number $11...1$ as $-1$.

This interpretation of negative numbers is called _two's complement_, and it is the most commonly used technique for representing negative numbers in computer systems. The idea is that the numbers from $0$ to $2^{N-1}-1$ are represented the way we are used to, with the most significant bit set to $0$. The binary strings with the most significant bit set to $1$ represent negative numbers:

* We start with $00...01$ representing $-2^{N-1}$, which is the smallest negative number representable in this system.
* From there, incrementing the binary bit string corresponds to incrementing the decimal number as well.
* Finally, $11...11$ always corresponds to $-1$.

You can think of this system as a variant of the unsigned arithmetic modulo $2^N$, but with a shifted range of representable numbers:

* In unsigned arithmetic, you can represent numbers from $0$ to $2^N - 1$, and when you increment $2^N - 1$, the result wraps around to $0$.
* In signed arithmetic, you can represent numbers from $-2^{N-1}$ to $2^{N-1} - 1$, and when you increment $2^{N-1} - 1$, the result wraps around to $-2^{N-1}$.

The following table gives an example of unsigned integers and signed integers in two's complement encoding for $N=3$ bits.

| Binary (little-endian) | Decimal (unsigned) | Decimal (signed) |
| :---: | :-: | :--: |
| $000$ | $0$ | $0$  |
| $100$ | $1$ | $1$  |
| $010$ | $2$ | $2$  |
| $110$ | $3$ | $3$  |
| $001$ | $4$ | $-4$ |
| $101$ | $5$ | $-3$ |
| $011$ | $6$ | $-2$ |
| $111$ | $7$ | $-1$ |

Mathematically, this can be written as a combination of a non-negative little-endian integer stored in the $N-1$ least significant bits and a $-2^{N-1}$ stored in the most significant bit (the sign bit):

$$
x_0x_1...x_{N-2}x_{N-1} = \underbrace{-x_{N-1} 2^{N-1}}_\text{sign bit} + \underbrace{x_{N-2} 2^{N-2} + \cdots + x_2 2^2 + x_1 2^1 + x_0 2^0}_{\text{unsigned } N-1 \text{-bit integer}}
$$

> The most negative number in two's complement notation, $00...01 = -2^{N-1}$, is a special case: it does not have a matching positive number in $N$-bit representation, and if we go through the steps of negating a number in two's complement notation (see next task), it will end up being its own negation!

So how exactly do we convert a positive number into a negative one (or vice versa)? Let's see!

### Problem 6. Negate the number

**Input:** 
A QInt register of length $N \ge 2$ in an arbitrary superposition state $\ket{a}$.

**Goal:** 
Negate the number $a$ stored in the register using two's complement, that is, transform the state to $\ket{-a}$. If $a = -2^{N-1}$, leave the state unchanged.

In [ ]:
from psiqdk.workbench import QInt
from test_ArithmeticDataTypes import problem

@problem
def negate(a: QInt) -> None:
    # Write your code here
    ...

### Problem 7. Add two signed integers of the same size

**Inputs:** 

1. A QInt register of length $N$ in an arbitrary superposition state $\ket{a}$.
2. A QInt register of length $N$ in an arbitrary superposition state $\ket{b}$.

**Goal:** 
Add the number $b$ to the number $a$ in two's complement notation (that is, making sure that the result is between $-2^{N-1}$ and $2^{N-1} - 1$, inclusive). Leave the register $\ket{b}$ unchanged.

In [ ]:
from psiqdk.workbench import Qubits, QInt, Qubrick
from test_ArithmeticDataTypes import problem

@problem
class NaiveAddSigned(Qubrick):
    def _increment(self, a: QInt, cond: Qubits) -> None:
        """Increment a QUInt register with a quantum control."""
        # Write your code here
        ...

    def _compute(self, a: QInt, b: QInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

### Problem 8. Add two signed integers of different sizes

**Inputs:** 

1. A QInt register of length $N$ in an arbitrary superposition state $\ket{a}$.
2. A QInt register of length $M \le N$ in an arbitrary superposition state $\ket{b}$.

**Goal:** 
Add the number $b$ to the number $a$ in two's complement notation (that is, making sure that the result is between $-2^{N-1}$ and $2^{N-1} - 1$, inclusive). Leave the register $\ket{b}$ unchanged.

<details>
<summary><strong>Need a hint?</strong></summary>
This time you can't just add two numbers, since their sign bits might be in different positions. You need to virtually "extend" the shorter number $b$ so that it is the same length as $a$ and their sign bits are aligned. But you have to do it without allocating any auxiliary qubits.
</details>

In [ ]:
from psiqdk.workbench import QInt
from test_ArithmeticDataTypes import problem

@problem
class NaiveAddSignedExtension(NaiveAddSigned):
    # The _increment method is inherited from NaiveAddSigned
    def _compute(self, a: QInt, b: QInt, **kwargs) -> None:
        """Add register b to register a in-place."""
        # Write your code here
        ...

Same as for unsigned subtraction, we can skip implementing signed subtraction as its own Qubrick, using adjoint of the addition instead.

## Exercise. Testing reversible computations - 2

Consider how you would test the Qubricks you've implemented in part 2. Can you reuse the test code from part 1 without modifications? If not, how do you need to modify it?

## Conclusion

Congratulations! In this kata you've learned to represent signed and unsigned integers and implement the naive addition algorithm.

> Copyright (c) 2026 PsiQuantum